In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

### Loading the dataset

In [ ]:
train_df = pd.read_csv('Google_Stock_Price_Train.csv')
train_df.info()

In [ ]:
test_df = pd.read_csv('Google_Stock_Price_Test.csv')
test_df.info()

#### Choosing column 'open' for predicition

In [ ]:
train = train_df.loc[:,["Open"]].values
train.shape

### Feature Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

In [ ]:
train_scaled = scaler.fit_transform(train)

In [ ]:
plt.plot(train_scaled)
plt.ylabel("Standardized Values")
plt.xlabel("Time->")
plt.show()

### Create data structure to train model  
Taking reference of past 60 days to predict future stock price  
x_train will have data of 60 days prior to current date and y_train will have price on current date

In [ ]:
x_train = []
y_train = []
time = 60
for i in range(60,train_scaled.shape[0]):
    x_train.append(train_scaled[i-60:i,0])
    y_train.append(train_scaled[i,0])
x_train = np.array(x_train)
y_train = np.array(y_train)

In [ ]:
x_train.shape,y_train.shape

In [ ]:
x_train = np.reshape(x_train,newshape=(x_train.shape[0],x_train.shape[1],1))
x_train.shape

### Build model

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN,Dropout

In [ ]:
model = Sequential()

model.add(SimpleRNN(units=50,activation = "tanh", return_sequences = True, input_shape = (x_train.shape[1], 1)))
model.add(Dropout(0.2))

model.add(SimpleRNN(units=50,activation = "tanh", return_sequences = True))
model.add(Dropout(0.2))

model.add(SimpleRNN(units=50,activation = "tanh", return_sequences = True))
model.add(Dropout(0.2))

model.add(SimpleRNN(units=50))
model.add(Dropout(0.2))

model.add(Dense(units=1))

model.compile(optimizer='adam',loss='mse')
model.summary()

In [ ]:
model.fit(x_train,y_train,epochs=100,batch_size=30,validation_split=0.05)

### Prepare test dataset

In [ ]:
data = pd.concat((train_df['Open'],test_df['Open']),axis=0)

In [ ]:
test_input = data.iloc[len(data) - len(test_df) - time : ].values
test_input.shape

In [ ]:
test_input = test_input.reshape(-1,1)
test_input.shape

In [ ]:
test_scaled = scaler.transform(test_input)

#### Create test data set

In [ ]:
x_test = []
for i in range(time,test_scaled.shape[0]):
    x_test.append(test_scaled[i - time: i,0 ])
x_test = np.array(x_test)
x_test.shape

In [ ]:
x_test = np.reshape(x_test,newshape=(x_test.shape[0],x_test.shape[1],1))
x_test.shape

In [ ]:
y_test = test_df.loc[:,"Open"].values

### Model Prediction

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
y_pred = scaler.inverse_transform(y_pred)

In [ ]:
output = model.evaluate(x=x_test,y=y_test)

In [ ]:
plt.plot(y_test, color = 'red', label = 'Real price')
plt.plot(y_pred, color = 'blue', label = 'Predicted price')

plt.title('Google Stock price prediction')
plt.xlabel('Time')
plt.ylabel('Price')
plt.legend()
plt.show()